# ForensicHub Forgery Detection Notebook

هذا الدفتر يوفر تدفق عمل كامل لتحميل وزن نموذج مدرَّب مسبقًا من ForensicHub، ورفع صور، ثم الحصول على درجة التلاعب بالإضافة إلى خريطة حرارية تبرز مناطق التعديل في حال توافرت أقنعة التنبؤ.

## 1. تثبيت المتطلبات وإعداد بيئة العمل

- إذا كنت تعمل على Kaggle أو بيئة جديدة، نفِّذ خلية التثبيت التالية مرة واحدة.
- إذا كان `ForensicHub` مثبتًا لديك بالفعل، يمكنك تعديل الخلية أو تخطيها.

In [ ]:
# اختياري: ثبّت المتطلبات في حال لم تكن موجودة بالفعل.
# أزِل أو عدّل هذه الخلية بما يتناسب مع مسار الحزمة لديك.
%pip install --quiet gdown ipywidgets matplotlib pillow torch torchvision

import sys
from pathlib import Path

REPO_ROOT = Path('.')  # عدِّل هذا المسار إذا كان مجلد ForensicHub في موقع آخر.
if str(REPO_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT.resolve()))

print('Python path configured. Current root:', REPO_ROOT.resolve())

## 2. تهيئة إعدادات النموذج

حرِّر القيم في الخلية التالية لتشير إلى وزن النموذج (الملف `*.pth` أو `*.pt`) الذي تريد استخدامه، بالإضافة إلى اسم النموذج كما هو مسجل في ForensicHub.

> **ملاحظة**: تأكد من أنك قمت برفع ملف الوزن إلى بيئتك (Kaggle Dataset أو رفع يدوي)، ثم حدِّد المسار الصحيح له.

In [ ]:
from pathlib import Path
import torch

CHECKPOINT_PATH = Path('/kaggle/input/your-model/checkpoint.pth')  # ← عدِّل هذا المسار
MODEL_NAME = 'ConvNextSmall'  # ← ضع اسم النموذج الصحيح المسجَّل في ForensicHub
IMAGE_SIZE = 512
DEVICE = 'auto'  # استخدم 'cpu' أو 'cuda' مباشرةً إذا أردت إجبار الجهاز
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
ADDITIONAL_INPUTS = {}  # استخدم هذه القيم إذا كان النموذج يحتاج مدخلات إضافية ثابتة

if DEVICE == 'auto':
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CHECKPOINT_PATH = CHECKPOINT_PATH.expanduser().resolve()
print(f'Using checkpoint: {CHECKPOINT_PATH}')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'لم يتم العثور على ملف الوزن: {CHECKPOINT_PATH}')
print(f'Running on device: {DEVICE}')

## 3. تحميل النموذج وتجهيز أدوات الاستدلال

يتم في الخلية التالية إنشاء فئة مساعده لتحميل النموذج وتنفيذ الاستدلال على الصور، بالإضافة إلى إنشاء قناع محلي ودمجه مع الصورة الأصلية.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, Iterable, Mapping, Optional

import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torchvision import transforms
from matplotlib import cm

from ForensicHub.registry import MODELS, build_from_registry


@dataclass
class InferenceConfig:
    model_name: str
    checkpoint_path: Path
    image_size: int
    device: str
    mean: Iterable[float]
    std: Iterable[float]
    additional_inputs: Optional[Mapping[str, Any]] = None


class ForgeryModelRunner:
    def __init__(self, config: InferenceConfig) -> None:
        self.config = config
        self.device = torch.device(config.device)
        self.model = self._load_model()
        self.transform = transforms.Compose([
            transforms.Resize((config.image_size, config.image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=config.mean, std=config.std),
        ])

    def _load_model(self) -> torch.nn.Module:
        model_kwargs = {
            'name': self.config.model_name,
            'init_path': str(self.config.checkpoint_path),
            'init_config': {'image_size': self.config.image_size},
        }
        model = build_from_registry(MODELS, model_kwargs)
        checkpoint = torch.load(self.config.checkpoint_path, map_location=self.device)
        state_dict = checkpoint.get('model', checkpoint)
        model.load_state_dict(state_dict)
        model.to(self.device).eval()
        return model

    def _prepare_inputs(self, image_tensor: torch.Tensor) -> Dict[str, Any]:
        inputs: Dict[str, Any] = {'image': image_tensor}
        if self.config.additional_inputs:
            for key, value in self.config.additional_inputs.items():
                tensor = torch.as_tensor(value)
                if tensor.ndim == 0:
                    tensor = tensor.unsqueeze(0)
                inputs[key] = tensor.to(self.device)
        return inputs

    def _extract_probability(self, outputs: Mapping[str, Any]) -> Optional[float]:
        prediction = outputs.get('pred_label')
        if prediction is None:
            prediction = outputs.get('pred')
        if prediction is None:
            return None
        tensor = torch.as_tensor(prediction).detach().float()
        if tensor.ndim == 0:
            tensor = tensor.unsqueeze(0)
        if tensor.ndim > 1:
            tensor = tensor.flatten()
        return float(torch.sigmoid(tensor[0]).cpu().item())

    def _extract_mask(self, outputs: Mapping[str, Any]) -> Optional[torch.Tensor]:
        mask = outputs.get('pred_mask')
        if mask is None:
            return None
        mask = torch.as_tensor(mask).detach().float()
        while mask.ndim < 4:
            mask = mask.unsqueeze(0)
        mask = mask[0]
        if mask.shape[0] > 1:
            mask = mask[0]
        mask = mask.squeeze()
        if mask.numel() == 0:
            return None
        if mask.min() < 0 or mask.max() > 1:
            mask = torch.sigmoid(mask)
        return mask.cpu()

    def _build_overlay(self, image: Image.Image, mask: torch.Tensor, alpha: float = 0.55):
        mask_tensor = F.interpolate(
            mask.unsqueeze(0).unsqueeze(0),
            size=image.size[::-1],
            mode='bilinear',
            align_corners=False,
        )[0, 0]
        mask_np = mask_tensor.numpy()
        mask_min, mask_max = float(mask_np.min()), float(mask_np.max())
        if mask_max > mask_min:
            mask_np = (mask_np - mask_min) / (mask_max - mask_min)
        mask_np = np.clip(mask_np, 0.0, 1.0)
        cmap = cm.get_cmap('magma')
        heatmap = cmap(mask_np)[..., :3]
        image_np = np.asarray(image).astype('float32') / 255.0
        overlay = (1 - alpha) * image_np + alpha * heatmap
        overlay = np.clip(overlay, 0.0, 1.0)
        overlay_img = Image.fromarray((overlay * 255).astype(np.uint8))
        return mask_np, overlay_img

    def predict(self, image: Image.Image) -> Dict[str, Any]:
        tensor = self.transform(image).unsqueeze(0).to(self.device)
        inputs = self._prepare_inputs(tensor)
        with torch.no_grad():
            outputs = self.model(**inputs)
        probability = self._extract_probability(outputs)
        mask_tensor = self._extract_mask(outputs)
        mask_np = None
        overlay_img = None
        if mask_tensor is not None:
            mask_np, overlay_img = self._build_overlay(image, mask_tensor)
        return {
            'probability': probability,
            'mask': mask_np,
            'overlay': overlay_img,
            'raw_outputs': outputs,
        }

    def predict_from_path(self, image_path: Path) -> Dict[str, Any]:
        image = Image.open(image_path).convert('RGB')
        result = self.predict(image)
        result['image_path'] = image_path
        return result

In [ ]:
config = InferenceConfig(
    model_name=MODEL_NAME,
    checkpoint_path=CHECKPOINT_PATH,
    image_size=IMAGE_SIZE,
    device=DEVICE,
    mean=MEAN,
    std=STD,
    additional_inputs=ADDITIONAL_INPUTS or None,
)
runner = ForgeryModelRunner(config)
print('Model ready for inference.')

## 4. واجهة تفاعلية لرفع الصور

تتيح لك الخلية التالية رفع صورة أو أكثر مباشرةً من جهازك ثم عرض النتيجة. يتم حساب:

- احتمالية أن تكون الصورة مزيفة.
- خريطة حرارية لمناطق التلاعب (إن وُجدت).

In [ ]:
import io
from typing import Dict

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt


output_area = widgets.Output()
file_upload = widgets.FileUpload(accept='image/*', multiple=True)
run_button = widgets.Button(description='تحليل الصور', button_style='primary', icon='search')


def render_results(original: Image.Image, prediction: Dict[str, Any]) -> None:
    prob = prediction.get('probability')
    overlay = prediction.get('overlay')

    if overlay is None:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(original)
        ax.set_title('الصورة الأصلية')
        ax.axis('off')
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(original)
        axes[0].set_title('الصورة الأصلية')
        axes[0].axis('off')
        axes[1].imshow(overlay)
        axes[1].set_title('مناطق التلاعب المتوقعة')
        axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    if prob is not None:
        print(f'احتمال كون الصورة مزيفة: {prob:.2%}')
    else:
        print('النموذج لم يرجع قيمة احتمالية (تحقق من إعدادات الإخراج).')


def on_run_clicked(_):
    with output_area:
        clear_output(wait=True)
        if not file_upload.value:
            print('الرجاء رفع صورة واحدة على الأقل.')
            return
        for name, file_info in file_upload.value.items():
            image = Image.open(io.BytesIO(file_info['content'])).convert('RGB')
            prediction = runner.predict(image)
            print('=' * 80)
            print(f'ملف: {name}')
            render_results(image, prediction)


run_button.on_click(on_run_clicked)

widgets.VBox([
    widgets.HTML('<h3>ارفع صورة ثم اضغط على زر "تحليل الصور"</h3>'),
    file_upload,
    run_button,
    output_area,
])

## 5. تحليل مجموعة صور من مجلد

إذا كان لديك مجلد يحتوي على مجموعة صور، يمكنك استخدام الخلية التالية لمعالجة جميع الصور دفعة واحدة، وتصدير النتائج إلى ملف CSV بالإضافة إلى حفظ الخرائط الحرارية على القرص (اختياري).

In [ ]:
from typing import Iterable
import pandas as pd

IMAGE_DIR = Path('/kaggle/input/your-images')  # ← عدِّل هذا المسار
SAVE_OVERLAYS = True
OVERLAY_DIR = Path('overlays')
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

records = []
if IMAGE_DIR.exists():
    image_paths: Iterable[Path] = sorted(
        p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.bmp'}
    )
    for image_path in image_paths:
        result = runner.predict_from_path(image_path)
        prob = result.get('probability')
        overlay = result.get('overlay')
        records.append({'image': image_path.name, 'probability': prob})

        if SAVE_OVERLAYS and overlay is not None:
            overlay_path = OVERLAY_DIR / f'{image_path.stem}_overlay.png'
            overlay.save(overlay_path)

    df = pd.DataFrame(records)
    display(df)
    df.to_csv('forgery_predictions.csv', index=False)
    print('تم حفظ النتائج في forgery_predictions.csv')
else:
    print('المجلد المحدد للصور غير موجود. قم بتعديله قبل التشغيل.')

## 6. نصائح إضافية
- إذا كان النموذج يتطلب مدخلات إضافية (مثل أقنعة أولية أو معرفات فئة)، يمكنك تمريرها عبر القاموس `ADDITIONAL_INPUTS`.
- بعض النماذج قد تُرجع مفاتيح مختلفة في الخرج. يمكنك فحص `result["raw_outputs"]` لمعرفة المحتويات بالتفصيل.
- لتسريع المعالجة على دفعات كبيرة، يمكنك تعديل الدالة `predict` لإرجاع النتائج على مجموعات (batch) من الصور.